# CHƯƠNG 11: Information Retrieval and Retrieval-Augmented Generation
**NHÓM 5:**

* **Mục đích:** Khắc phục các hạn chế cốt lõi của Mô hình ngôn ngữ lớn (LLM) bao gồm: hiện tượng ảo tưởng thông tin (hallucination), tri thức bị đóng băng sau khi huấn luyện (static knowledge) và không có khả năng truy cập dữ liệu nội bộ/bảo mật. Hệ thống tích hợp một bộ truy xuất dữ liệu ngoài nhằm cung cấp thông tin có căn cứ (grounded facts) giúp LLM sinh câu trả lời chính xác.
* **Bộ dữ liệu (Dataset):** Sử dụng WikiQA (phát hành bởi Microsoft), một bộ dữ liệu benchmark chuẩn hóa gồm các câu hỏi thực tế và các đoạn văn bản trích xuất từ Wikipedia. Việc sử dụng benchmark này giúp đánh giá năng lực của hệ thống RAG ở quy mô thực tế thay vì dữ liệu tự tạo.
* **Thuật toán & Kiến trúc:** * **Retriever (Bộ truy xuất dày đặc - Dense Retrieval):** Sử dụng mô hình nhúng dạng Bi-Encoder chuyển văn bản thành các dense vectors. Kết hợp thư viện FAISS (Facebook AI Similarity Search) để tính tương đồng Cosine qua phép toán Inner Product nhằm tìm kiếm thực thể lân cận gần nhất (KNN) với tốc độ tối ưu.
  * **Generator (Bộ tạo câu trả lời):** Sử dụng mô hình Flan-T5-Base. Đây là kiến trúc Encoder-Decoder (Text-to-Text) đã được tinh chỉnh qua câu lệnh hướng dẫn (Instruction-tuned), giúp mô hình tuân thủ nghiêm ngặt ngữ cảnh đầu vào và giảm thiểu tối đa hiện tượng ảo tưởng.

**BƯỚC 1: CÀI ĐẶT VÀ KHỞI TẠO THƯ VIỆN**
* Tiến hành cài đặt các thư viện mã nguồn mở cần thiết: `transformers` để chạy LLM, `sentence-transformers` để tạo vector nhúng, và `faiss-cpu` để tăng tốc tìm kiếm không gian đa chiều.

In [18]:
!pip install -q transformers sentence-transformers faiss-cpu datasets ipywidgets

import faiss
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 73.1 MB/s eta 0:00:00


**BƯỚC 2: TẢI DATASET BENCHMARK VÀ XÂY DỰNG CHỈ MỤC VECTOR (FAISS)**
* Hệ thống tải tập dữ liệu `wiki_qa` thực tế để trích xuất 3,000 đoạn văn bản làm kho tri thức (Corpus).
* Các văn bản thô được chuyển hóa thành các vector dày đặc (Dense Vectors) qua mô hình nhúng `all-MiniLM-L6-v2`. Sau đó, các vector được chuẩn hóa dạng $L_2$ để đưa về phép toán Inner Product (`IndexFlatIP`), tương đương với việc tính tương đồng Cosine.

In [19]:
print("--- ĐANG TẢI DATASET CHUẨN TỪ HUGGING FACE ---")
dataset = load_dataset("wiki_qa", split="test")

# Trích xuất và loại bỏ trùng lặp ngữ cảnh
raw_corpus = [item['answer'] for item in dataset if item['answer'] != ""]
corpus = list(set(raw_corpus))[:3000]
print(f"Đã tải thành công! Quy mô hệ thống: {len(corpus)} đoạn văn bản từ Wikipedia.")

print("\n--- TIẾN HÀNH MÃ HÓA VÀ NẠP VÀO CHỈ MỤC FAISS ---")
retriever_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
corpus_embeddings = retriever_model.encode(corpus)

# Cấu hình Index dựa trên số chiều không gian của vector nhúng
dimension = corpus_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

# Chuẩn hóa L2 để tính Cosine Similarity thông qua phép toán tích vô hướng
faiss.normalize_L2(corpus_embeddings)
index.add(corpus_embeddings)
print(f"Chỉ mục FAISS đã sẵn sàng với {index.ntotal} dense vectors.")

--- ĐANG TẢI DATASET CHUẨN TỪ HUGGING FACE ---
Đã tải thành công! Quy mô hệ thống: 3000 đoạn văn bản từ Wikipedia.

--- TIẾN HÀNH MÃ HÓA VÀ NẠP VÀO CHỈ MỤC FAISS ---


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chỉ mục FAISS đã sẵn sàng với 3000 dense vectors.


**BƯỚC 3: ĐỊNH NGHĨA GIAI ĐOẠN TRUY XUẤT VÀ TẠO PROMPT (Retrieval Stage)**
* Xây dựng hàm thực hiện truy xuất thông tin ngữ nghĩa (Semantic Search). Khi câu hỏi được nhập vào, nó sẽ được chuyển thành vector nhúng, FAISS tiến hành so khớp để trích xuất ra $k=5$ đoạn văn bản có điểm số (Score) cao nhất.
* Xây dựng cấu trúc Prompt ràng buộc (Instruction Prompt) để ép mô hình ngôn ngữ chỉ được trả lời dựa trên những gì được cung cấp.

In [20]:
def retrieve_relevant_passages(query, k=5):
    query_embedding = retriever_model.encode([query])
    faiss.normalize_L2(query_embedding)

    # Tìm kiếm KNN trong không gian vector nhúng
    scores, indices = index.search(query_embedding, k)
    retrieved_docs = [corpus[idx] for idx in indices[0]]

    # Định dạng chuỗi hiển thị log trực quan
    log_output = f"<b>[Truy xuất] Đã tìm thấy {len(retrieved_docs)} đoạn văn bản liên quan nhất:</b><br>"
    for i, doc in enumerate(retrieved_docs):
        log_output += f"&nbsp;&nbsp;- Đoạn {i+1} (Score: {scores[0][i]:.4f}): {doc}<br>"

    return retrieved_docs, log_output

def build_prompt(query, context_docs):
    context_str = "\n".join([f"- {doc}" for doc in context_docs])
    prompt = (
        f"Answer the question based on the provided context accurately. "
        f"If the context does not contain the answer, reply with 'unanswerable'.\n\n"
        f"Context:\n{context_str}\n\n"
        f"Question: {query}\n"
        f"Answer:"
    )
    return prompt

**BƯỚC 4: KHỞI TẠO MÔ HÌNH GENERATOR VÀ ĐỊNH NGHĨA LUỒNG SINH VĂN BẢN**
* **Khởi tạo mô hình `google/flan-t5-base`**: Mô hình này áp dụng kiến trúc **Encoder-Decoder** giúp phân tích sâu chuỗi ngữ cảnh đầu vào (ở Encoder) và chắt lọc ký tự đầu ra (ở Decoder).
* **Cấu hình `temperature=0.2`**: Nhằm hạn chế tính ngẫu nhiên của thuật toán lấy mẫu (sampling), giúp câu trả lời có tính ổn định cao, chính xác và bám sát văn bản gốc.
* **Cấu hình `max_length=128`**: Giới hạn độ dài tối đa của câu trả lời sinh ra là 128 tokens. Việc này giúp tối ưu hóa thời gian tính toán, ngăn mô hình bị lặp từ vô hạn và hoàn toàn phù hợp với các câu hỏi sự thật (Factoid QA) vốn có đáp án rất cô đọng.

In [21]:
print("--- ĐANG TẢI MÔ HÌNH GENERATOR (FLAN-T5-BASE) ---")
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
generator_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print("Đã tải xong mô hình bộ tạo câu trả lời.")

def generate_answer(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = generator_model.generate(**inputs, max_length=128, temperature=0.2)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

--- ĐANG TẢI MÔ HÌNH GENERATOR (FLAN-T5-BASE) ---


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Đã tải xong mô hình bộ tạo câu trả lời.


**BƯỚC 5: DEMO INTERACTIVE TRỰC QUAN (Giao diện nhập câu hỏi thực tế)**
* Để hệ thống hoạt động trực quan phục vụ cho quá trình nghiệm thu bài tập, ô mã nguồn dưới đây thiết lập một giao diện tương tác động bằng `ipywidgets`. Bạn có thể nhập câu hỏi bất kỳ để quan sát luồng hoạt động từ bước Truy xuất (Retrieval) đến bước Sinh câu trả lời (Generation).

In [24]:
# Thiết lập các phần tử giao diện
query_input = widgets.Text(
    value='What is the official fiat currency of the United States?',
    placeholder='Nhập câu hỏi tại đây...',
    description='Câu hỏi:',
    layout=widgets.Layout(width='60%')
)
run_button = widgets.Button(
    description='Chạy Hệ Thống RAG',
    button_style='success',
    icon='play'
)
output_area = widgets.Output()

def on_button_clicked(b):
    with output_area:
        output_area.clear_output()
        query = query_input.value

        # 1. Thực hiện giai đoạn truy xuất văn bản
        relevant_contexts, retrieval_log = retrieve_relevant_passages(query, k=5)

        # 2. Tạo prompt và đẩy qua LLM sinh câu trả lời
        prompt = build_prompt(query, relevant_contexts)
        answer = generate_answer(prompt)

        # 3. Hiển thị kết quả ra giao diện bằng HTML định dạng đẹp
        display(HTML(f"""
            <div style='padding:10px; background-color:#e8f4fd; border-left:5px solid #28a745; color:#1a1a1a; font-family:sans-serif;'>
                {retrieval_log}
            </div>
        """))

        display(HTML(f"""
            <br><b style='color:inherit;'>[PROMPT GỬI ĐẾN LLM]:</b>
            <pre style='background:#f4f6f8; padding:10px; border:1px solid #ddd; color:#222222; white-space:pre-wrap; font-family:monospace;'>{prompt}</pre>
        """))

        display(HTML(f"""
            <div style='font-size:16px; color:#d9534f; margin-top:10px; font-family:sans-serif;'>
                <b style='color:#d9534f;'>=> CÂU TRẢ LỜI CỦA HỆ THỐNG RAG:</b> <span style='color:inherit; font-weight:bold;'>{answer}</span>
            </div>
        """))

run_button.on_click(on_button_clicked)

# Hiển thị giao diện điều khiển lên Colab
display(widgets.VBox([widgets.HBox([query_input, run_button]), output_area]))